In [ ]:
import numpy as np
import pandas as pd
import yaml
import os
from matplotlib import pyplot as plt
from hsa_hopper.kinematics import *
from hsa_hopper.controller import HopController
from hsa_hopper.hsa_model import HSAModel
actuator_model_path = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\notebooks\actuator_model.yaml'
with open(actuator_model_path, 'r') as f:
    actuator_model = yaml.load(f, yaml.Loader)

char_model_path = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\notebooks\hsa_model.yaml'
with open(char_model_path, 'r') as f:
    hsa_char_model = HSAModel.make_from_dict(yaml.load(f, yaml.Loader))
kin_params = KinematicParameters(.07,.15,.3,-.005)

def load_data(root, include_dynamics_params=True):
    dataframes = {}
    hardware_configs = {}
    dynamics_params = {}
    controller_params = {}

    for category in os.listdir(root):
        print(category)
        dataframes[category] = {}
        hardware_configs[category] = {}
        dynamics_params[category] = {}
        controller_params[category] = {}
        category_folder = os.path.join(root,category)
        for trial in os.listdir(category_folder):
            trial_folder = os.path.join(category_folder, trial)
            trial_df = pd.read_csv(os.path.join(trial_folder,'data.csv'))
            dataframes[category][trial] = trial_df
            # load config files
            with open(os.path.join(trial_folder,'hardware_config.yaml')) as f:
                hardware_configs[category][trial] = yaml.load(f,yaml.Loader)
            with open(os.path.join(trial_folder,'experiment_config.yaml')) as f:
                experiment_config = yaml.load(f,yaml.Loader)
                servo_pos = experiment_config['controller']['servo_pos']
                trial_df['psi'] = (servo_pos-1500)*(np.pi/4)*200
                if include_dynamics_params:
                    dynamics_params[category][trial] = experiment_config['dynamics_params']
                    m_foot = dynamics_params[category][trial]['m_foot']
                    m_cart = dynamics_params[category][trial]['m_cart'] 
                    total_mass = m_cart+m_foot
                    trial_df['total_mass'] = m_cart+m_foot 
                else:
                    total_mass = int(category.split('_')[-1][:-1])/1000
                    m_foot = actuator_model['m_foot']
                    trial_df['total_mass'] = total_mass
                    m_cart = total_mass - m_foot
                controller_params[category][trial] = experiment_config['controller']
            # load (smoothed) data files
    return dataframes, hardware_configs, dynamics_params, controller_params


In [177]:
aim1_path = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\data\design_paper'
aim1_data = load_data(aim1_path, include_dynamics_params = False)
aim2_path = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\data\aim2\hopping_data'
aim2_data = load_data(aim2_path, include_dynamics_params = True)

hsa_1300g
1.3
hsa_1500g
1.5
hsa_1700g
1.7
hsa_1900g
1.9
hsa_2100g
2.1
no_hsa_1300g
1.3
no_hsa_1500g
1.5
no_hsa_1700g
1.7
no_hsa_1900g
1.9
no_hsa_2100g
2.1
hsa_1300g
hsa_1500g
hsa_1700g
hsa_1900g
hsa_2100g
no_hsa_1300g
no_hsa_1500g
no_hsa_1700g
no_hsa_1900g
no_hsa_2100g


In [180]:
R = 1.5*.095
gear_ratio = 6
Km = (105) # 105 RPM per Volt
Kt = gear_ratio*(3/2)/np.sqrt(3)*(60/(2*np.pi))/Km # torque constant in Nm/A or Volts * seconds
print(Kt)
# run these on each trials dataframe individually - do not concatenate separate trials

# do power calculations and add them to the data frame
def power_calcs(df: pd.DataFrame):
    x = df['x_rad'].to_numpy()
    xdot = df['xdot'].to_numpy()
    xddot = df['xddot'].to_numpy()
    ydot = df['ydot'].to_numpy()
    yddot = df['yddot'].to_numpy()
    tau = df['torque'].to_numpy()
    total_mass = df['total_mass'].to_numpy()
    mode = df['mode'].to_numpy()
    m_cart = total_mass - actuator_model['m_foot']
    unique_mode = df['mode'].unique()
    if len(unique_mode) == 3:
        # this data is from the design paper
        leg_inertia = ((mode==2)+(mode==3))*m_cart+(mode==1)*actuator_model['m_foot']
    else:
        # this data is from the modeling paper
        leg_inertia = (mode == HopController._STANCE)*m_cart+(mode == HopController._FLIGHT)*actuator_model['m_foot']
    therm_power = R*(tau/Kt)**2
    joint_power = actuator_model['J']*xddot*xdot + leg_inertia*(yddot+9.81)*ydot
    motor_power = tau*xdot
    elec_power = therm_power + motor_power
    df['therm_power'] = therm_power
    df['joint_power'] = joint_power
    df['motor_power'] = motor_power
    df['elec_power'] = elec_power

def energy_calcs(df: pd.DataFrame):
    hop_idx = np.sort(df['hop_idx'].unique())
    therm_work = np.zeros(len(hop_idx)-1)
    joint_work = np.zeros_like(therm_work)
    motor_work = np.zeros_like(therm_work)
    elec_work = np.zeros_like(therm_work)
    hop_height = np.zeros_like(therm_work)
    COT = np.zeros_like(therm_work)
    TCOT = np.zeros_like(therm_work)
    JCOT = np.zeros_like(therm_work)
    MCOT = np.zeros_like(therm_work)
    unique_modes = df['mode'].unique()
    for i, idx in enumerate(hop_idx[:-1]):
        frame = df['hop_idx'] == idx
        t = df.loc[frame,'t_s'].to_numpy()
        P = df.loc[frame,'therm_power'].to_numpy()
        therm_work[i] = np.trapz(P,t)
        P = df.loc[frame,'joint_power'].to_numpy()
        joint_work[i] = np.trapz(P,t)
        P = df.loc[frame,'motor_power'].to_numpy()
        motor_work[i] = np.trapz(P,t)
        P = df.loc[frame,'elec_power'].to_numpy()
        elec_work[i] = np.trapz(P,t)
        y = df.loc[frame, 'y_m'].to_numpy()
        # x_raw = df.loc[frame, 'x_raw'].to_numpy()
        # fk,jac = forward_kinematics(kin_params, x_raw, jacobian=True)
        ydot = df.loc[frame, 'ydot'].to_numpy()
        t = df.loc[frame, 't_s'].to_numpy()
        mode = df.loc[frame, 'mode'].to_numpy()
        # if len(unique_modes) == 3:
        #     # this data is from the design paper
        #     flight_start = next(i for i in range(len(mode)) if mode[i] == 1)
        # else:
        flight_start = next(i for i in range(len(mode)) if mode[i] == HopController._FLIGHT)
            # this data is from the modeling paper
        # xdot = (x_raw[flight_start]-x_raw[flight_start-1])/(t[flight_start]-t[flight_start-1])
        # ydot = (y[flight_start-1]-y[flight_start-2])/(t[flight_start-1]-t[flight_start-2])
        # ydot = jac[0]*xdot
        # hop_height[i] = (ydot**2)/(2*9.81)
        hop_height[i] = (ydot[flight_start-1]**2)/(2*9.81)
        total_mass = df.loc[frame,'total_mass'].to_numpy()[0]
        cost_norm = total_mass*9.81*hop_height[i]
        COT[i] = elec_work[i]/cost_norm
        TCOT[i] = therm_work[i]/cost_norm
        JCOT[i] = joint_work[i]/cost_norm
        MCOT[i] = motor_work[i]/cost_norm
    return pd.DataFrame({
        'hop_idx': hop_idx[:-1],
        'therm_work': therm_work,
        'joint_work' : joint_work,
        'motor_work' : motor_work,
        'elec_work' : elec_work,
        'hop_height' : hop_height,
        'COT' : COT,
        'TCOT' : TCOT,
        'JCOT' : JCOT,
        'MCOT' : MCOT,
    })

aim1_energy = {}
for condition, trials in aim1_data[0].items():
    aim1_energy[condition] = {}
    for trial, df in trials.items():
        power_calcs(df)
        aim1_energy[condition][trial] = energy_calcs(df)
aim2_energy = {}
for condition, trials in aim2_data[0].items():
    aim2_energy[condition] = {}
    for trial, df in trials.items():
        power_calcs(df)
        aim2_energy[condition][trial] = energy_calcs(df)

0.47256762464725044


In [181]:
for condition, trials in aim1_energy.items():
    for trial, result in trials.items():
        print(condition)
        hop_height_mean = result['hop_height'].mean()
        cot_mean = result['COT'].mean()
        print(f'average hop height: {hop_height_mean}')
        print(f'average COT: {cot_mean}')

hsa_1300g
average hop height: 0.05345418207262241
average COT: 2.3321191493202855
hsa_1500g
average hop height: 0.05235945164354259
average COT: 2.328533357029375
hsa_1700g
average hop height: 0.05238163429947712
average COT: 2.1610269091698306
hsa_1900g
average hop height: 0.05408614093264427
average COT: 2.1495517590350524
hsa_2100g
average hop height: 0.05813062127273753
average COT: 2.033980791997533
no_hsa_1300g
average hop height: 0.05482027182779689
average COT: 2.4197164929013306
no_hsa_1500g
average hop height: 0.0534362830989541
average COT: 2.484976355615885
no_hsa_1700g
average hop height: 0.052614359103006804
average COT: 2.580129215190303
no_hsa_1900g
average hop height: 0.051576773954259025
average COT: 2.649108297746731
no_hsa_2100g
average hop height: 0.05292263517814299
average COT: 2.796413547632867


In [ ]:
for condition, trials in aim2_energy.items():
    for trial, result in trials.items():
        print(condition)
        hop_height_mean = result['hop_height'].mean()
        cot_mean = result['COT'].mean()
        print(f'average hop height: {hop_height_mean}')
        print(f'average COT: {cot_mean}')

{'hsa_1300g': {'2025-04-14_1744666247':     hop_idx  therm_work  joint_work  motor_work  elec_work  hop_height  \
  0         0    0.207750   -0.041331    0.999583   1.207333    0.038709   
  1         1    0.205784    0.035507    0.944951   1.150735    0.039505   
  2         2    0.218341    0.058728    0.974740   1.193080    0.040255   
  3         3    0.226053    0.057961    1.013700   1.239753    0.041016   
  4         4    0.224052    0.071690    1.007999   1.232051    0.041323   
  5         5    0.222185    0.075021    0.998205   1.220390    0.040512   
  6         6    0.225290    0.047075    1.002428   1.227717    0.039458   
  7         7    0.228887    0.071822    1.001565   1.230452    0.040073   
  8         8    0.218888   -0.000780    1.014086   1.232974    0.039593   
  9         9    0.208336    0.129618    0.951894   1.160230    0.041270   
  10       10    0.228438    0.081156    1.024355   1.252794    0.041507   
  11       11    0.225407    0.098724    0.983683 